In [1]:
import os

from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import FastEmbedSparse, QdrantVectorStore, RetrievalMode
from pydantic import SecretStr
from qdrant_client.models import FieldCondition, Filter, MatchValue

c:\Artificial Intelligence\rag_tutorial\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

api_key = SecretStr(os.getenv("OPENROUTER_API_KEY", ""))
if not api_key.get_secret_value():
    raise RuntimeError("OPENROUTER_API_KEY is not set")

In [3]:
embedding_model = OpenAIEmbeddings(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1",
    model="openai/text-embedding-3-small",
)

In [4]:
texts = [
    "Neuronale Netze können komplexe Muster lernen",
    "Gradientenabstieg optimiert Modellparameter",
    "Overfitting tritt auf, wenn Modelle Trainingsdaten auswendig lernen",
    "RAG kombiniert Retrieval mit Generation",
]

In [5]:
vectorstore = QdrantVectorStore.from_texts(
    texts=texts,
    embedding=embedding_model,
    collection_name="ml_course_docs",
    location=":memory:",
)

In [6]:
results = vectorstore.similarity_search("Was ist Overfitting?", k=2)
for r in results:
    print(r)

page_content='Overfitting tritt auf, wenn Modelle Trainingsdaten auswendig lernen' metadata={'_id': '3ab016a4-e732-4fe0-9ab9-c814932fef33', '_collection_name': 'ml_course_docs'}
page_content='Gradientenabstieg optimiert Modellparameter' metadata={'_id': '15080f8b-b741-4129-8e93-4897c34d72b1', '_collection_name': 'ml_course_docs'}


In [7]:
for i, doc in enumerate(results, 1):
    print(f"{i}. {doc.page_content}")

1. Overfitting tritt auf, wenn Modelle Trainingsdaten auswendig lernen
2. Gradientenabstieg optimiert Modellparameter


In [8]:
irrelevant_results = vectorstore.similarity_search_with_score(
    "Was ist das Rezept für Schokoladenkuchen?", k=2
)

In [9]:
for doc, score in irrelevant_results:
    print(f"[{score:.3f}] {doc.page_content}")

[0.186] Gradientenabstieg optimiert Modellparameter
[0.145] RAG kombiniert Retrieval mit Generation


### Semantische Suche


In [10]:
docs = [
    "The bug introduced in commit a3f7bc2 caused the tokenizer to silently drop special tokens during batch encoding.",
    "Git tracks changes through a directed acyclic graph of commits, where each commit stores a snapshot of the repository state along with metadata.",
    "Continuous integration runs automated tests on every commit to catch regressions before they reach the main branch.",
    "Debugging machine learning pipelines often requires bisecting commits to find where model performance regressed.",
    "Cherry-picking individual commits allows developers to selectively apply bug fixes from one branch to another without merging all changes.",
    "The git blame command shows which commit last modified each line of a file, helping developers understand the history behind code changes.",
    "Squashing multiple commits into one creates a cleaner history by combining related changes into a single commit before merging.",
]

In [11]:
semantic_store = QdrantVectorStore.from_texts(
    texts=docs,
    embedding=embedding_model,
    collection_name="semantic_demo",
    location=":memory:",
)

In [12]:
results = semantic_store.similarity_search_with_relevance_scores(
    "Can I integrate individual changes from other versions of the software?", k=2
)
for doc, score in results:
    print(f"[{score:.2f}] {doc.page_content}")

[0.76] Cherry-picking individual commits allows developers to selectively apply bug fixes from one branch to another without merging all changes.
[0.72] Continuous integration runs automated tests on every commit to catch regressions before they reach the main branch.


### Semantische Suche versagt bei Commit-Hashes

In [13]:
results_hash = semantic_store.similarity_search_with_relevance_scores("commit a3f7bc2", k=2)
for doc, score in results_hash:
    print(f"[{score:.2f}] {doc.page_content}")


[0.72] Squashing multiple commits into one creates a cleaner history by combining related changes into a single commit before merging.
[0.70] Cherry-picking individual commits allows developers to selectively apply bug fixes from one branch to another without merging all changes.


### Die Lösung: Hybride Suche

In [14]:
sparse_embedding_model = FastEmbedSparse(model_name="Qdrant/bm25")

In [15]:
hybrid_store = QdrantVectorStore.from_texts(
    texts=docs,
    embedding=embedding_model,
    sparse_embedding=sparse_embedding_model,
    collection_name="hybrid_demo",
    location=":memory:",
    retrieval_mode=RetrievalMode.HYBRID,
)

In [16]:
query = "commit a3f7bc2"
results_semantic = semantic_store.similarity_search_with_relevance_scores(query, k=2)
for doc, score in results_semantic:
    print(f"[{score:.2f}] {doc.page_content[:80]}...")

[0.72] Squashing multiple commits into one creates a cleaner history by combining relat...
[0.70] Cherry-picking individual commits allows developers to selectively apply bug fix...


In [17]:
results_semantic_query = hybrid_store.similarity_search_with_relevance_scores(
    "Can I integrate individual changes from other versions of the software?", k=2
)
for doc, score in results_semantic_query:
    print(f"[{score:.2f}] {doc.page_content[:80]}...")  

[1.00] Cherry-picking individual commits allows developers to selectively apply bug fix...
[0.83] Continuous integration runs automated tests on every commit to catch regressions...


### Dokumente mit Metadaten

In [18]:
my_docs = [
    Document(page_content="Der Mount Everest ist mit 8849 Metern der höchste Berg der Erde",
             metadata={"category": "geography"}),
    Document(page_content="Der Kilimandscharo ist der höchste Berg Afrikas mit 5895 Metern",
             metadata={"category": "geography"}),
    Document(page_content="Die Alpen erstrecken sich über acht europäische Länder",
             metadata={"category": "geography"}),
    Document(page_content="Bergsteigen erfordert gute Kondition und spezielle Ausrüstung",
             metadata={"category": "activity"}),
    Document(page_content="Der K2 gilt als schwierigster Achttausender der Welt",
             metadata={"category": "geography"}),
    Document(page_content="Wandern in den Bergen ist eine beliebte Freizeitaktivität",
             metadata={"category": "activity"}),
]

In [19]:
# Dense Vektor-Store erstellen, semantisch suchen
my_store = QdrantVectorStore.from_documents(
    documents=my_docs,
    embedding=embedding_model,
    collection_name="workshop_docs",
    location=":memory:",
)

In [20]:
test_queries = ["höchste Berge", "Outdoor-Aktivitäten", "mountains in Europe"]
for q in test_queries:
    results = my_store.similarity_search(q, k=2)
    print(f"\nQuery: '{q}'")
    for doc in results:
        print(f"  [{doc.metadata['category']}] {doc.page_content}")


Query: 'höchste Berge'
  [geography] Der Mount Everest ist mit 8849 Metern der höchste Berg der Erde
  [geography] Der Kilimandscharo ist der höchste Berg Afrikas mit 5895 Metern

Query: 'Outdoor-Aktivitäten'
  [activity] Wandern in den Bergen ist eine beliebte Freizeitaktivität
  [activity] Bergsteigen erfordert gute Kondition und spezielle Ausrüstung

Query: 'mountains in Europe'
  [geography] Die Alpen erstrecken sich über acht europäische Länder
  [activity] Wandern in den Bergen ist eine beliebte Freizeitaktivität


In [21]:
print("Relevante Anfrage:")
results_relevant = my_store.similarity_search_with_score("höchste Berge", k=2)
for doc, score in results_relevant:
    print(f"  [{score:.3f}] {doc.page_content}")

Relevante Anfrage:
  [0.545] Der Mount Everest ist mit 8849 Metern der höchste Berg der Erde
  [0.523] Der Kilimandscharo ist der höchste Berg Afrikas mit 5895 Metern


Vektor-Datenbanken geben **immer** Ergebnisse zurück — auch bei irrelevanten
Anfragen. Führen Sie eine relevante und eine irrelevante Anfrage aus,
vergleichen Sie die Scores und wählen Sie einen Schwellenwert, der die
irrelevante Anfrage herausfiltert.

In [22]:
print("\nIrrelevante Anfrage:")
results_irrelevant = my_store.similarity_search_with_score(
    "Wie programmiert man in Python?", k=2
)
for doc, score in results_irrelevant:
    print(f"  [{score:.3f}] {doc.page_content}")


Irrelevante Anfrage:
  [0.091] Die Alpen erstrecken sich über acht europäische Länder
  [0.074] Wandern in den Bergen ist eine beliebte Freizeitaktivität


In [23]:
print("Ergebnisse filtern:")
threshold = 0.5

print(f"Mit Schwellenwert {threshold}:")
for doc, score in results_irrelevant:
    status = "RELEVANT" if score >= threshold else "GEFILTERT"
    print(f"  [{score:.3f}] {status}: {doc.page_content}")

Ergebnisse filtern:
Mit Schwellenwert 0.5:
  [0.091] GEFILTERT: Die Alpen erstrecken sich über acht europäische Länder
  [0.074] GEFILTERT: Wandern in den Bergen ist eine beliebte Freizeitaktivität


### Nach Kategorie filtern + hybride Suche

1. Bauen Sie einen **hybriden** Vektor-Store (dense + sparse/BM25) aus denselben Dokumenten
2. Suchen Sie nach einer Anfrage mit einem spezifischen Begriff (`"K2"`) — vergleichen Sie semantisch vs. hybride
3. Nutzen Sie einen Metadaten-**Filter**, um die Suche auf eine `category` einzuschränken

In [24]:
my_hybrid_store = QdrantVectorStore.from_documents(
    documents=my_docs,
    embedding=embedding_model,
    sparse_embedding=sparse_embedding_model,
    collection_name="workshop_hybrid",
    location=":memory:",
    retrieval_mode=RetrievalMode.HYBRID,
)

In [25]:
exact_query = "K2"
filter_query = "mountains"

In [26]:
print("Semantische Suche (nur dense):")
for doc in my_store.similarity_search(exact_query, k=2):
    print(f"  [{doc.metadata['category']}] {doc.page_content}")

print("\nHybride Suche (dense + BM25):")
for doc in my_hybrid_store.similarity_search(exact_query, k=2):
    print(f"  [{doc.metadata['category']}] {doc.page_content}")

Semantische Suche (nur dense):
  [geography] Der K2 gilt als schwierigster Achttausender der Welt
  [geography] Der Kilimandscharo ist der höchste Berg Afrikas mit 5895 Metern

Hybride Suche (dense + BM25):
  [geography] Der K2 gilt als schwierigster Achttausender der Welt
  [geography] Der Kilimandscharo ist der höchste Berg Afrikas mit 5895 Metern


### Nach Kategorie filtern

`"mountains"` passt auf beide Kategorien. Nutzen Sie einen
Metadaten-**Filter**, um die hybride Suche auf die Kategorie `"activity"`
einzuschränken

In [27]:
activity_filter = Filter(
    must=[FieldCondition(key="metadata.category", match=MatchValue(value="activity"))]
)

In [28]:
print(f"Query: '{filter_query}'\n")

print("Ohne Filter:")
for doc in my_hybrid_store.similarity_search(filter_query, k=3):
    print(f"  [{doc.metadata['category']}] {doc.page_content}")

print("\nNur Kategorie activity:")
for doc in my_hybrid_store.similarity_search(filter_query, k=3, filter=activity_filter):
    print(f"  [{doc.metadata['category']}] {doc.page_content}")

Query: 'mountains'

Ohne Filter:
  [activity] Wandern in den Bergen ist eine beliebte Freizeitaktivität
  [geography] Der Kilimandscharo ist der höchste Berg Afrikas mit 5895 Metern
  [geography] Der Mount Everest ist mit 8849 Metern der höchste Berg der Erde

Nur Kategorie activity:
  [activity] Wandern in den Bergen ist eine beliebte Freizeitaktivität
  [activity] Bergsteigen erfordert gute Kondition und spezielle Ausrüstung
